# 04 — Memory with engram_lite

Conversations don't fit in context windows. The naive fix is to truncate — drop the oldest messages when the window fills. The actual fix is to **decide what's worth keeping**.

That's what memory systems do: they maintain context across turns without exceeding the token budget. `engram_lite` is a minimal implementation — JSONL storage, text-based scoring, and clear rules about what gets retrieved.

## The problem: context overflow

Start with a working system from notebook 02:

In [2]:
from llm_engines import get_engine
from llm_engines.contracts import GenerationRequest

engine = get_engine("ollama", "qwen3:8b")

# Simulate a conversation
history = [
    {"role": "user", "content": "I'm working on a Python web scraper."},
    {"role": "assistant", "content": "Great! What site are you scraping?"},
    {"role": "user", "content": "A job board. I need to extract job titles and salaries."},
    {"role": "assistant", "content": "Use BeautifulSoup for parsing. What format is the data in?"},
]

# Add a new turn
request = GenerationRequest(
    messages=history + [{"role": "user", "content": "The salary is in a <span class='salary'>."}],
    temperature=0.7,
)

response = engine.generate(request)
print(response.message.content[:200])

Got it. Here’s a minimal example using **BeautifulSoup** to extract job titles and salaries from a job board where salaries are inside `<span class='salary'>`:

```python
from bs4 import BeautifulSoup


That works — the model sees the full conversation and gives a relevant answer.

Now simulate 50 turns. At some point you hit the context limit (typically 128K tokens, ~100K words). What happens?

**Option 1: Truncate from the front.** Drop old messages to make room. Problem: the model loses critical context. 'I'm working on a web scraper' gets dropped, and by turn 50 the model doesn't remember what you're building.

**Option 2: Memory.** Keep the important parts, drop the filler.

## Memory as context engineering

`engram_lite` stores conversation history in JSONL files and uses **context engineering** — deciding what to surface given a token budget.

Two components:
- **Recent turns:** Last N turns from the session, always included for continuity
- **Episode search:** Scores past turns based on text similarity to the current query

When you query memory, it ranks candidates by:
- **Lexical overlap** — token matching between query and stored text
- **Recency** — newer turns score higher
- **Importance** — user-flagged or heuristic-detected important turns
- **Specificity** — distinctive terms get a boost

It assembles context by ranking candidates and filling the budget. **No vector embeddings, no graph database** — just fast text scoring against JSONL files.

## Engram_lite vs full engram

This course uses `engram_lite` - a minimal, file-based memory system. It is designed for learning: 
explicit storage calls, visible JSONL files, straightforward text scoring.

**Full `engram`** (the production system in your toolkit) is more sophisticated:

- **5 memory layers:** Working (SQLite), Episodic (ChromaDB with vector embeddings), Semantic (Kuzu graph), 
  Cold storage (SQLite FTS5), Neural (RTRL weights)
- **Automatic assistant processing:** Pairs assistant responses with preceding user turns and stores complete 
  Q&A exchanges, not isolated messages
- **Background daemon:** Async event processing for ingestion, extraction, and forgetting
- **Semantic extraction:** LLM-based fact extraction to populate the knowledge graph
- **Neural memory:** RTRL layer learns user -> assistant response patterns

The core concepts are the same (context assembly, scoring, token budgeting), but `engram` handles 
much of the curation automatically. You decide *what* to remember with `engram_lite`; `engram` 
decides *how* to represent and retrieve it.

For this course, the simplicity of `engram_lite` is a feature - you see every decision explicitly. 
In production, the automation in engram is worth the complexity.

## Discover the API

Before using `engram_lite`, check what it exports:

In [3]:
import engram_lite

exports = [name for name in dir(engram_lite) if not name.startswith('_')]
print("Exports:", exports)

# You should see: ProjectMemory, AugmentRequest, AugmentResult, PromptAugmenter
# The main class is ProjectMemory.

Exports: ['AugmentRequest', 'AugmentResult', 'ProjectMemory', 'PromptAugmenter', 'augment_result_to_interop_result', 'contracts', 'describe_memory', 'inspection', 'interop', 'memory', 'project_memory', 'prompting', 'retrieval', 'trace_to_memory_records', 'version']


## ProjectMemory basics

`ProjectMemory` manages conversation history for one project. Initialize it with a storage path:

In [4]:
from engram_lite import ProjectMemory
import tempfile
from pathlib import Path

# Create a temporary directory for this demo
demo_dir = Path(tempfile.mkdtemp())

memory = ProjectMemory(
    base_dir=demo_dir,
    project_id="scraper_demo",
    session_id="session_1",
    total_prompt_tokens=2000,  # token budget for assembled context
)

print(f"Memory storage: {demo_dir / 'scraper_demo'}")
print(f"Episodes file: {memory._episodes_path}")
print(f"Sessions dir: {memory._sessions_dir}")

Memory storage: /tmp/tmpfun1sirw/scraper_demo
Episodes file: /tmp/tmpfun1sirw/scraper_demo/episodes.jsonl
Sessions dir: /tmp/tmpfun1sirw/scraper_demo/sessions


The memory creates:
- `scraper_demo/episodes.jsonl` — all stored turns
- `scraper_demo/sessions/session_1.jsonl` — this session's turns

Everything is human-readable JSONL. You can inspect it with `cat` or a text editor.

## Writing turns to memory

Store conversation turns as they happen:

In [6]:
# Write conversation turns to the session
memory.add_turn(
    role="user",
    text="I'm building a web scraper for job boards.",
    session_id="session_1",
)

memory.add_turn(
    role="assistant",
    text="Great! What site are you targeting?",
    session_id="session_1",
)

memory.add_turn(
    role="user",
    text="Indeed.com. Need to extract job titles and salaries.",
    session_id="session_1",
)

memory.add_turn(
    role="assistant",
    text="Use BeautifulSoup to parse the HTML. What format is the salary data in?",
    session_id="session_1",
)

# For explicit importance scoring, use store_episode instead:
memory.store_episode(
    text="Decision: Use BeautifulSoup for parsing job board HTML",
    importance=0.9,  # high importance
    metadata={"topic": "web_scraping", "type": "decision"},
)

print(f"Stored {len(memory._episodes)} episodes")
print(f"Session turns: {len(memory._sessions.get('session_1', []))}")

Stored 1 episodes
Session turns: 4


**Importance scores** (0.0-1.0) affect retrieval ranking. The system automatically detects some important turns (user preferences, decisions, explicit memory requests), but you can override with explicit scores.

## Retrieving context

When you query memory, it assembles context from recent turns + scored episode search:

In [7]:
# Query memory
result = memory.build_prompt(
    user_message="What was the scraper project about again?",
    query="scraper project",
    max_prompt_tokens=2000,
    return_trace=True,
)

print("=== Assembled prompt ===")
print(result["prompt"][:500])
print(f"\nPrompt tokens: {result['prompt_tokens']}")
print(f"Memory tokens: {result['memory_tokens']}")

=== Assembled prompt ===
## Working
1. user: I'm building a web scraper for job boards.
2. assistant: Great! What site are you targeting?
3. user: Indeed.com. Need to extract job titles and salaries.
4. assistant: Use BeautifulSoup to parse the HTML. What format is the salary data in?

## Episodic
1. Use BeautifulSoup for parsing job board HTML.

## User
What was the scraper project about again?

Prompt tokens: 64
Memory tokens: 46


The prompt now includes relevant past turns. The model can answer 'web scraper for job boards' even though that was mentioned several turns ago.  Note: The assistant statement in episodic memory was stored explicitly using store_episode().  Engram remembers both user and assistant outputs automatically.

## Inspecting with traces

Use `llm_inspector` from notebook 03 to see what memory retrieved:

In [19]:
from llm_inspector import render_comparison

# The trace is in result['trace']
trace = result.get('trace')

if trace:
    print(f"Sections retrieved: {len(trace.sections)}")
    for i, section in enumerate(trace.sections):
        print(f"\nSection {i}: {section.title}")
        print(f"  Origin: {section.origin}")
        print(f"  Tokens: {section.tokens}")
        print(f"  Text: {section.text[:100]}...")
    
    print(f"\nToken accounting:")
    print(f"  Total tokens: {trace.token_accounting.total_tokens}")
    print(f"  Compressed: {trace.token_accounting.compressed}")
    print(f"  Truncated: {trace.token_accounting.truncated}")
else:
    print("No trace available. Set return_trace=True in build_prompt().")

Sections retrieved: 4

Section 0: Working
  Origin: working
  Tokens: 45
  Text: 1. user: I'm building a web scraper for job boards.
2. assistant: Great! What site are you targeting...

Section 1: Episodic
  Origin: episodic
  Tokens: 10
  Text: 1. Use BeautifulSoup for parsing job board HTML....

Section 2: User
  Origin: user
  Tokens: 9
  Text: What was the scraper project about again?...

Section 3: Final prompt
  Origin: prompt
  Tokens: 64
  Text: ## Working
1. user: I'm building a web scraper for job boards.
2. assistant: Great! What site are yo...

Token accounting:
  Total tokens: 64
  Compressed: False
  Truncated: False


The trace shows:
- Which past turns were retrieved
- How they were scored
- Token accounting (how much budget each section consumed)

This is essential for debugging. If the model gives a bad answer, check the trace — did memory surface the wrong context?

## The advisor pattern in memory

A key optimization that can be applied: **use a fast model for candidates, a slow model for judgment**.

In `engram_lite`, the scoring is text-based (no LLM calls), so it's already fast. But when you build on top of this — say, using an LLM to extract facts or judge relevance — you'd apply the advisor pattern:

1. **Fast model** (e.g., `qwen3:8b`) generates summaries of retrieved turns
2. **Slow model** (e.g., `claude-sonnet`) judges which summaries are relevant
3. Return the filtered set

This is the foundation for the full advisor pattern in notebook 09. Here it's implicit in the design — cheap scoring first, expensive operations only on top candidates.

## Thinking vs non-thinking modes

Another useful concept: some operations benefit from chain-of-thought, others don't.

**If you use an LLM for memory operations:**

**Enable thinking for:**
- Judging relevance ('Is this turn about the scraper project?')
- Extracting facts ('What did we decide about the API?')
- Deciding what to forget ('Is this turn still relevant?')

**Disable thinking for:**
- Generating embeddings (if you add vector search)
- Formatting retrieved context (template filling)
- Simple lookups ('Get the last 5 turns')

`engram_lite` itself doesn't call LLMs for retrieval — it's text scoring. But when you extend it (adding an LLM-based fact extractor, for example), toggle thinking mode per operation.

From notebook 00, you already know how: `reasoning_effort: 'none'` disables it, omitting that parameter enables it.

## Failure mode: irrelevant retrieval

Memory fails when it surfaces irrelevant context. Simulate it:

In [23]:
# Add an unrelated conversation to the same project
memory.add_turn(role="user", text="Also, what's a good restaurant in SF?", session_id="session_2")
memory.add_turn(role="assistant", text="Try Zuni Café.", session_id="session_2")

# Now query about the scraper
result = memory.build_prompt(
    user_message="How do I parse the salary data?",
    query="parse salary",
    max_prompt_tokens=2000,
    return_trace=True,
)

trace = result['trace']
print("Retrieved sections:")
for section in trace.sections:
    if 'restaurant' in section.text.lower() or 'zuni' in section.text.lower():
        print(f"  ⚠ IRRELEVANT: {section.text[:80]}")
    else:
        print(f"  ✓ {section.text[:80]}")

Retrieved sections:
  ⚠ IRRELEVANT: 1. user: Also, what's a good restaurant in SF?
2. assistant: Try Zuni Café.
3. u
  ✓ 1. Use BeautifulSoup for parsing job board HTML.
  ✓ How do I parse the salary data?
  ⚠ IRRELEVANT: ## Working
1. user: Also, what's a good restaurant in SF?
2. assistant: Try Zuni


If the restaurant turn appears, scoring failed — the text similarity wasn't smart enough to filter it out. How to fix:
- **Better scoring:** Add topic modeling or keyword extraction
- **Explicit filtering:** Tag turns with topics, filter before scoring
- **LLM-based relevance:** Use a small model to judge 'is this about web scraping?' before including in context

The trace makes this visible. Without it, you'd just see 'model gave a weird answer' and have no idea why.

## Token budgeting

Memory operates under a budget. If you allocate 2000 tokens:

- System prompt: 200 tokens (fixed)
- Recent turns: 500 tokens (last 3-5 turns)
- Retrieved episodes: 1000 tokens (scored search results)
- User message: 100 tokens
- Reserve for output: 200 tokens

Total: 2000. If retrieval returns more than 1000 tokens, truncate the lowest-scored items.

The trace shows the accounting — `token_accounting.per_origin_budget` breaks it down.

## Artifact: memory-augmented chat

Build a chat function that maintains context across turns:

In [26]:
def chat_with_memory(
    message: str,
    memory: ProjectMemory,
    engine,
) -> str:
    """
    Chat with memory-augmented context.
    
    Args:
        message: User's message
        memory: ProjectMemory instance
        engine: LLM engine from llm_engines
    
    Returns:
        Model response
    """
    # Build prompt with memory
    result = memory.build_prompt(
        user_message=message,
        query=message,
        max_prompt_tokens=2000,
        return_trace=False,  # disable for production
    )
    
    prompt_text = result['prompt']
    
    # Convert to messages format
    # engram_lite returns a text prompt; convert to messages
    messages = [{"role": "user", "content": prompt_text}]
    
    # Generate response
    from llm_engines.contracts import GenerationRequest
    request = GenerationRequest(messages=messages, temperature=0.7)
    response = engine.generate(request)
    
    # Store the turn for next time
    memory.add_turn(role="user", text=message, session_id="session_3")
    memory.add_turn(role="assistant", text=response.message.content, session_id="session_3")
    
    return response.message.content

# Test it
response = chat_with_memory(
    "What was I building again?",
    memory,
    engine,
)
print(response[:200])

Based on the conversation history provided, it looks like you were previously discussing **restaurants in San Francisco** (specifically Zuni Café) and **parsing job board HTML using BeautifulSoup**.




## What's next

You now understand:
- Memory as context engineering (not blind storage)
- How `engram_lite` scores and retrieves (text similarity, recency, importance)
- Token budgeting across memory sources
- The advisor pattern foundation (fast scoring, slow judgment)
- Thinking vs non-thinking modes for LLM-based memory operations

Notebook 05 applies these same ideas to **RAG** — retrieval-augmented generation. Instead of conversation turns, you're retrieving documents. The mechanics are identical: search, rank, assemble within budget, trace to verify.

## Exercises

1. **Explore the storage.** After running the examples, inspect the JSONL files at `{demo_dir}/scraper_demo/`. Read `episodes.jsonl` and see how turns are stored. What fields does each episode have?

2. **Simulate irrelevant retrieval.** Add 20 turns about different topics to the same memory. Query for one topic. Use traces to see if irrelevant turns were retrieved. What made them score high enough to be included?

3. **Token budget stress test.** Write 100 turns, then query with a 1000-token budget. Check `token_accounting` in the trace — what got truncated first? Why?

4. **Build a multi-session assistant.** Create a `ProjectMemory`, have two different sessions each with 10 turns. Query from one session — does it see turns from the other session? (It should, because episodes are project-level, not session-level.)

---
**Next:** [05 — RAG with rag_lib](05_rag_with_rag_lib.ipynb) uses similar retrieval mechanics for documents instead of conversation turns, with chunking strategies from the production RAG article.